# 18.4 变分推断 / Variational Inference (ELBO & Mean-Field)

**中文**：MCMC(18.3)精确但慢——要跑成千上万步、还得诊断收敛。当数据海量、模型复杂时,采样慢得无法接受。**变分推断(Variational Inference, VI)** 换了个思路:*"与其从后验里艰难采样，不如把推断变成一个**优化问题**——从一个简单的分布族里,挑一个最接近真后验的 $q$。"* VI 把"抽样"换成"优化",于是能用 SGD、能上大数据、能 GPU 加速。代价是**近似有偏**。VI 是现代概率机器学习(包括 **VAE**)的引擎。
**English**: MCMC (18.3) is exact but slow — thousands of steps plus convergence diagnostics. With massive data and complex models, sampling becomes unacceptably slow. **Variational Inference (VI)** takes a different route: *"instead of laboriously sampling the posterior, turn inference into an **optimization problem** — from a simple family of distributions, pick the $q$ closest to the true posterior."* VI replaces "sampling" with "optimization," enabling SGD, big data, and GPUs. The cost is a **biased approximation**. VI is the engine of modern probabilistic ML (including the **VAE**).

---

**中文**：VI 要找一个简单分布 $q(\theta)$ 去逼近真后验 $p(\theta|\mathcal D)$，"接近"用 **KL 散度**衡量。直接最小化 $\text{KL}(q\|p)$ 需要那个算不出的证据 $p(\mathcal D)$。妙处在于:最小化 KL **等价于**最大化一个能算的量——**证据下界(ELBO, Evidence Lower BOund)**:
**English**: VI seeks a simple $q(\theta)$ to approximate the true posterior $p(\theta|\mathcal D)$, with "closeness" measured by **KL divergence**. Directly minimizing $\text{KL}(q\|p)$ needs the intractable evidence $p(\mathcal D)$. The trick: minimizing KL is **equivalent** to maximizing a computable quantity — the **ELBO (Evidence Lower BOund)**:

$$\log p(\mathcal D)=\underbrace{\text{ELBO}(q)}_{\text{要最大化}}+\underbrace{\text{KL}(q\|p)}_{\ge 0}\ \Rightarrow\ \text{ELBO}(q)=\mathbb E_{q}[\log p(\mathcal D,\theta)]-\mathbb E_q[\log q(\theta)]$$

**中文**：因为 $\log p(\mathcal D)$ 是常数、KL≥0,所以**最大化 ELBO = 最小化 KL = 让 q 最接近后验**。ELBO 又可拆成"**期望对数似然(拟合数据)− KL(q‖先验)(别偏离先验太远)**"——天然平衡拟合与正则。
**English**: Since $\log p(\mathcal D)$ is constant and KL≥0, **maximizing the ELBO = minimizing KL = making q closest to the posterior**. The ELBO decomposes into "**expected log-likelihood (fit the data) − KL(q‖prior) (don't stray from the prior)**" — naturally balancing fit and regularization.

**中文**：最常用的简化叫 **平均场(mean-field)**:假设 $q$ **在各变量上因子分解** $q(\theta)=\prod_i q_i(\theta_i)$(各变量独立)。这让优化可解(常有闭式坐标上升 CAVI),但也埋下了 VI 的**两大天生偏差**,本节会亲手揭示。
**English**: The most common simplification is **mean-field**: assume $q$ **factorizes over variables** $q(\theta)=\prod_i q_i(\theta_i)$ (variables independent). This makes optimization tractable (often closed-form coordinate ascent, CAVI) but plants VI's **two inherent biases**, which we expose hands-on.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 现代贝叶斯必考）**
> **中文**：VI=把推断变成优化——用简单 q 逼近后验, **最大化 ELBO(=最小化 KL(q‖p), 绕开证据)**。ELBO=期望对数似然 − KL(q‖先验)。**平均场**=q 因子分解(变量独立), 用坐标上升(CAVI)或**随机VI/黑盒VI(重参数化梯度, 就是 VAE 的做法)**优化。**两大天生偏差(必考)**:①**低估方差/过度自信**——平均场砍掉变量相关性, q 比真后验更"瘦";②**mode-seeking(寻众数)**——KL(q‖p) 是"零强制"的, 多峰后验时 q 会**塌到单个峰**而非覆盖全部(对比 KL(p‖q) 的 mean-seeking)。**vs MCMC**:VI 快/可扩展/确定性但**有偏**; MCMC 慢但**渐近精确**。
> **English**: VI = inference as optimization — approximate the posterior with a simple q, **maximize the ELBO (= minimize KL(q‖p), bypassing the evidence)**. ELBO = expected log-likelihood − KL(q‖prior). **Mean-field** = q factorizes (variables independent), optimized by coordinate ascent (CAVI) or **stochastic/black-box VI (reparameterization gradients — exactly what the VAE does)**. **Two inherent biases (interview favorites)**: ① **variance underestimation / overconfidence** — mean-field drops variable correlations, so q is "thinner" than the true posterior; ② **mode-seeking** — KL(q‖p) is "zero-forcing," so for multimodal posteriors q **collapses to a single mode** rather than covering all (vs KL(p‖q)'s mean-seeking). **vs MCMC**: VI is fast/scalable/deterministic but **biased**; MCMC is slow but **asymptotically exact**.


In [ ]:

# ============================================================
# 偏差一:平均场低估方差 / Bias #1: mean-field underestimates variance
# 中文:目标后验是一个【相关】二维高斯 N(0, [[1,ρ],[ρ,1]])。平均场假设 q(x)q(y) 独立(无相关)。
#      用坐标上升(CAVI)优化。看它砍掉相关性后, 方差被压得多小。
# English: target posterior is a CORRELATED 2D Gaussian N(0, [[1,ρ],[ρ,1]]). Mean-field assumes
#      q(x)q(y) independent (no correlation). Optimize by coordinate ascent (CAVI).
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
np.random.seed(0)
rho=0.85
Sigma=np.array([[1,rho],[rho,1]]); Prec=np.linalg.inv(Sigma)  # 目标精度矩阵 / target precision

# 平均场 CAVI:对高斯目标, 最优 q_i 的方差 = 1/Prec_ii (条件方差), 均值互相耦合
# For a Gaussian target, optimal mean-field q_i variance = 1/Prec_ii (conditional variance!)
vx=vy=1.0/Prec[0,0]                                          # 平均场最优方差=条件方差 / conditional var
mx=my=0.0
def kl_qp(mx,my,vx,vy):                                      # KL(q||p), 两个高斯间闭式 / analytic KL
    mq=np.array([mx,my]); Sq=np.diag([vx,vy])
    return 0.5*(np.trace(Prec@Sq)+mq@Prec@mq-2+np.log(np.linalg.det(Sigma)/np.linalg.det(Sq)))
kl_hist=[kl_qp(mx,my,vx,vy)]
for _ in range(30):                                         # CAVI 坐标上升(此对称目标均值收敛到0)/ CAVI
    mx=-Prec[0,1]/Prec[0,0]*my; my=-Prec[1,0]/Prec[1,1]*mx
    kl_hist.append(kl_qp(mx,my,vx,vy))
print(f"真后验边际方差 / true marginal variance: {Sigma[0,0]:.3f}")
print(f"平均场 q 的方差 / mean-field q variance: {vx:.3f}  (= 条件方差 1-ρ² = {1-rho**2:.3f})")
print(f"→ 低估了 {(1-vx/Sigma[0,0])*100:.0f}%! 平均场砍掉相关性, q 比真后验瘦一圈")
print(f"→ underestimated by {(1-vx/Sigma[0,0])*100:.0f}%! mean-field is overconfident")


**中文**：平均场把边际方差从 1.0 压到了 $1-\rho^2$——**严重低估、过度自信**。根源:真后验里 $x,y$ 强相关(斜椭圆),但平均场强行假设它们独立,只能用一个**轴对齐的、更瘦的**椭圆去贴,自然把"沿相关方向的展开"给砍没了。下面可视化,并展示第二个偏差:**mode-seeking(寻众数)**。
**English**: Mean-field squeezes the marginal variance from 1.0 to $1-\rho^2$ — **severe underestimation, overconfidence**. The cause: in the true posterior $x,y$ are strongly correlated (a tilted ellipse), but mean-field forces independence and can only fit an **axis-aligned, thinner** ellipse, cutting off the spread along the correlation direction. Next we visualize this and expose the second bias: **mode-seeking**.


In [ ]:

# ============================================================
# 偏差二:mode-seeking —— 用单个高斯逼近双峰目标 / Bias #2: mode-seeking on a bimodal target
# 中文:目标 p = 0.5·N(-2,0.6) + 0.5·N(2,0.6)(双峰)。用【单个高斯 q】逼近。
#      VI 最小化 KL(q||p)("反向KL", zero-forcing) → q 会塌到【一个峰】; 而矩匹配(正向KL)会跨坐两峰。
# English: target p = 0.5·N(-2,0.6)+0.5·N(2,0.6) (bimodal). Approximate with a SINGLE Gaussian q.
#      VI minimizes KL(q||p) ("reverse KL", zero-forcing) → q collapses to ONE mode; moment-matching
#      (forward KL) straddles both.
# ============================================================
def target_pdf(z): return 0.5*stats.norm.pdf(z,-2,0.6)+0.5*stats.norm.pdf(z,2,0.6)

# 反向 KL(q||p)(VI 用的): 网格搜最优单高斯 / reverse KL used by VI: grid-search optimal single Gaussian
def reverse_kl(m,s):
    z=np.random.default_rng(0).normal(m,s,40000)
    logq=stats.norm.logpdf(z,m,s); logp=np.log(target_pdf(z)+1e-300)
    return np.mean(logq-logp)
best=(1e9,None)
for m in np.linspace(-3,3,61):
    for s in np.linspace(0.3,2.5,45):
        k=reverse_kl(m,s)
        if k<best[0]: best=(k,(m,s))
rk_m,rk_s=best[1]

# 正向 KL(p||q)(矩匹配): 最优 q = 匹配 p 的均值和方差 / forward KL: match moments of p
fk_m=0.0                                                    # E_p[z] = 0 (对称) / mean of p
fk_var=0.6**2 + 2.0**2                                      # Var_p = 分量方差 + 分量均值方差 / mixture var
fk_s=np.sqrt(fk_var)
print(f"反向 KL(q||p) 最优(VI):    m={rk_m:.2f}, s={rk_s:.2f}  → 塌到一个峰(mode-seeking)")
print(f"正向 KL(p||q) 最优(矩匹配): m={fk_m:.2f}, s={fk_s:.2f}  → 跨坐两峰(mean-seeking)")
print("这就是 VI(反向KL)的天性:宁可精准覆盖一个峰, 也不含糊地摊开覆盖所有峰")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.8))
# ① 平均场低估方差:真后验(相关) vs q(轴对齐更瘦)/ mean-field underestimates variance
g=np.linspace(-3,3,120); gx,gy=np.meshgrid(g,g); pos=np.dstack([gx,gy])
ax[0].contour(gx,gy,stats.multivariate_normal([0,0],Sigma).pdf(pos),colors="#C44E52",linewidths=1.5)
ax[0].contour(gx,gy,stats.multivariate_normal([mx,my],np.diag([vx,vy])).pdf(pos),colors="#4C72B0",linewidths=1.5)
ax[0].plot([],[],"#C44E52",label="真后验(相关) true"); ax[0].plot([],[],"#4C72B0",label="平均场 q(过度自信)")
ax[0].set_title("偏差1:平均场低估方差 / underestimates variance"); ax[0].legend(fontsize=8); ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
# ② CAVI 的 KL 单调下降 / CAVI KL monotonically decreases
ax[1].plot(kl_hist,"o-",color="#55A868")
ax[1].set_title("CAVI:KL(q‖p) 单调下降(=ELBO上升)/ ELBO increases"); ax[1].set_xlabel("坐标上升迭代"); ax[1].set_ylabel("KL(q‖p)")
ax[1].axhline(kl_hist[-1],ls=":",color="gray")
# ③ mode-seeking:反向KL 塌到一峰, 正向KL 跨两峰 / mode-seeking vs mean-seeking
zz=np.linspace(-5,5,400)
ax[2].fill_between(zz,target_pdf(zz),color="gray",alpha=0.3,label="双峰目标 p")
ax[2].plot(zz,stats.norm.pdf(zz,rk_m,rk_s),"#4C72B0",lw=2,label="VI 反向KL(塌到一峰)")
ax[2].plot(zz,stats.norm.pdf(zz,fk_m,fk_s),"#DD8452",lw=2,ls="--",label="矩匹配 正向KL(跨两峰)")
ax[2].set_title("偏差2:VI mode-seeking / mode-seeking"); ax[2].set_xlabel("z"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/bay04_viz.png",dpi=80); plt.show()
print("VI(反向KL)选择精准贴一个峰; 若你要覆盖所有峰, VI 会误导你")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **VI 把推断变成优化,快得多**:不用采样、不用等收敛,一路梯度/坐标上升最大化 ELBO 即可。CAVI 的 KL 单调下降到收敛(中图)。这正是 VI 能上大数据、能用 GPU、能塞进深度学习(VAE 的编码器就是在做 amortized VI)的原因。
2. **但 VI 有两大天生偏差,务必牢记**:
   - **低估方差/过度自信**(左图):平均场假设变量独立,砍掉了真后验的相关性,q 比真后验**瘦一圈**(本例边际方差被低估了约 70%)。所以 **VI 给的不确定性往往偏小、偏乐观**——在需要可靠不确定性的场景(风控、医疗)要格外警惕。
   - **mode-seeking(右图)**:VI 最小化的反向 KL(q‖p) 是"零强制"的——它极度惩罚"在 p 很小处 q 却很大",于是宁可**精准塌进一个峰**,也不敢摊开去覆盖所有峰。如果真后验是多峰的(如混合模型、神经网络),**VI 可能只报告一个峰,漏掉其他解**。对比:正向 KL(矩匹配)会跨坐所有峰(但会在峰间的低概率区放很多质量)。
3. **没有免费的午餐:VI 快 vs MCMC 准**:VI 用"简单分布族 + 优化"换来速度和可扩展性,代价是**系统性的偏差**(方差偏小、可能漏峰)。MCMC 慢但渐近精确。实践中:大数据/深度模型用 VI(甚至 VAE),要精确后验/小数据用 MCMC。**知道 VI 会低估不确定性、会 mode-seeking,是用好它的前提。**

**English**:
1. **VI turns inference into optimization — far faster**: no sampling, no waiting for convergence, just gradient/coordinate ascent on the ELBO. CAVI's KL decreases monotonically to convergence (middle). This is why VI scales to big data, uses GPUs, and fits into deep learning (a VAE's encoder does amortized VI).
2. **But VI has two inherent biases — always remember them**:
   - **Variance underestimation / overconfidence** (left): mean-field assumes independence, cutting the true posterior's correlation, so q is **thinner** than the true posterior (here the marginal variance is underestimated by ~70%). So **VI's uncertainty is often too small, too optimistic** — beware in settings needing reliable uncertainty (risk, medicine).
   - **Mode-seeking** (right): the reverse KL(q‖p) that VI minimizes is "zero-forcing" — it heavily penalizes "q large where p is small," so it would rather **collapse precisely onto one mode** than spread to cover all. For a multimodal true posterior (mixtures, neural nets), **VI may report only one mode, missing other solutions**. Contrast: forward KL (moment matching) straddles all modes (but places mass in low-probability valleys between them).
3. **No free lunch: VI fast vs MCMC accurate**: VI trades a "simple family + optimization" for speed and scalability, at the cost of **systematic bias** (too-small variance, possible missed modes). MCMC is slow but asymptotically exact. In practice: big-data / deep models use VI (even the VAE); accurate-posterior / small-data use MCMC. **Knowing that VI underestimates uncertainty and is mode-seeking is a prerequisite for using it well.**

> 💼 **实战视角 / Practical angle**
> **中文**:VI 是**可扩展贝叶斯**与深度概率模型的引擎:①**VAE**——编码器输出 $q(z|x)$ 的均值方差, 用重参数化技巧做黑盒 VI(这就是 Part 13 学的);②**贝叶斯深度学习**(权重不确定性, Bayes by Backprop);③PyMC/NumPyro 的 **ADVI**(自动微分变分推断)——一键把任意模型变成 VI。**工程要点**:①监控 ELBO 收敛;②记住不确定性会偏小(可用更灵活的 q, 如**归一化流 normalizing flow** 缓解);③多峰问题慎用。面试金句:*"VI 用简单 q 逼近后验、最大化 ELBO(=最小化反向KL, 绕开证据); 快且可扩展(VAE 就是), 但平均场低估方差、反向KL mode-seeking——这两个偏差是 VI 的天性, 也是与 MCMC 的核心权衡。"*
> **English**: VI is the engine of **scalable Bayesian** and deep probabilistic models: ① the **VAE** — the encoder outputs the mean/variance of $q(z|x)$, doing black-box VI via the reparameterization trick (Part 13); ② **Bayesian deep learning** (weight uncertainty, Bayes by Backprop); ③ PyMC/NumPyro's **ADVI** (automatic differentiation VI) — turn any model into VI with one line. **Engineering**: ① monitor ELBO convergence; ② remember uncertainty is too small (use a more flexible q, e.g. **normalizing flows**, to mitigate); ③ be cautious with multimodality. Interview line: *"VI approximates the posterior with a simple q, maximizing the ELBO (= minimizing reverse KL, bypassing the evidence); fast and scalable (the VAE is VI), but mean-field underestimates variance and reverse-KL is mode-seeking — these biases are VI's nature and the core trade-off vs MCMC."*

---
### 小结 / Summary
- **中文**:VI=把推断变优化, 用简单 q 逼近后验, 最大化 ELBO(=最小化 KL(q‖p), 绕开证据)。
- **English**: VI = inference as optimization, approximate the posterior with a simple q, maximize the ELBO (= minimize KL(q‖p), bypass the evidence).
- **中文**:两大天生偏差:平均场低估方差(过度自信)、反向KL mode-seeking(多峰塌到单峰)。
- **English**: Two inherent biases: mean-field underestimates variance (overconfident), reverse-KL is mode-seeking (collapses multimodal to one mode).
- **中文**:VI 快/可扩展(VAE、ADVI)但有偏; MCMC 慢但精确——按数据规模与精度需求取舍。
- **English**: VI is fast/scalable (VAE, ADVI) but biased; MCMC is slow but exact — choose by data scale and accuracy needs.
